In [51]:
import pandas as pd
import os
import scipy.stats as stats

# ----------------------------
# PARTE 1: IDENTIFICAZIONE E CATEGORIZZAZIONE DELLE VARIABILI
# ----------------------------

# Percorso del file con i dati dei pazienti
data_file = r"C:\Users\Lorenzo\Desktop\scoring/demographics/summary_extracted.xlsx"
df = pd.read_excel(data_file)

df['Group'] = df['Group'].astype('category')

# Suddivisione delle variabili in base al loro tipo
continuous_vars = ['Age', 'Disease_Duration', 'LEDD']
ordinal_vars = ['UPDRSIV4.1','UPDRSI', 'UPDRSII', 'UPDRSIII', 'UPDRSIV', 'AIMs', 'Hoen_Year']
categorical_vars = ['Gender', 'DA_AGONIST_YES1NOT0', 'BDZ_YES1NOT0']

# Otteniamo i gruppi presenti nell'ordine desiderato
desired_order = ['CTL', 'DNV', 'ADV', 'DYS']
groups = [g for g in desired_order if g in df['Group'].unique()]

# ----------------------------
# Modifica dell'analisi per 'Disease_Duration' e 'LEDD' per escludere 'CTL'
# ----------------------------

# Le variabili per cui non vogliamo includere 'CTL'
exclude_ctl_vars = ['Disease_Duration', 'LEDD']

cont_results = []
for var in continuous_vars:
    # Controllo per escludere il gruppo "CTL" per alcune variabili
    if var in exclude_ctl_vars:
        groups_without_ctl = [grp for grp in groups if grp != 'CTL']
    else:
        groups_without_ctl = groups

    result_row = {}
    result_row['Variable'] = var
    result_row['Type'] = 'Continuous'

    # --- Normalità (Test Shapiro-Wilk per ogni gruppo) ---
    normality_pvalues = {}
    decisions = [] 
    sufficient_groups = []  # Tiene traccia dei gruppi con dati sufficienti
    
    for grp in groups_without_ctl:
        data_grp = df[df['Group'] == grp][var].dropna()
        if len(data_grp) < 3:
            normality_pvalues[grp] = "n<3"
            decisions.append("Insufficient")
        else:
            sufficient_groups.append(grp)
            if data_grp.var() == 0:
                normality_pvalues[grp] = "zero variance"
                decisions.append("Non-normal")
            else:
                try:
                    stat, p = stats.shapiro(data_grp)
                    normality_pvalues[grp] = round(p, 3)
                    decisions.append("Normal" if p >= 0.05 else "Non-normal")
                except Exception as e:
                    normality_pvalues[grp] = f"Error: {str(e)}"
                    decisions.append("Error")
    
    # Verifica gruppi sufficienti
    if len(sufficient_groups) < 2:
        overall_normality = "Insufficient groups with n≥3"
        variance_status = "Insufficient groups with n≥3"
        levene_p = "NA"
        suggested_test = "Insufficient data for statistical testing"
    else:
        if all(dec == "Insufficient" for dec in decisions):
            overall_normality = "Insufficient data"
        elif any(dec == "Non-normal" for dec in decisions if dec != "Insufficient"):
            overall_normality = "Non-normal"
        else:
            overall_normality = "Normal"
        
        # --- Omogeneità delle varianze (Test di Levene) ---
        samples = []
        for grp in sufficient_groups:
            data_grp = df[df['Group'] == grp][var].dropna()
            samples.append(data_grp.values)
        
        if len(samples) < 2:
            variance_status = "Insufficient data"
            levene_p = "NA"
        else:
            try:
                stat_levene, p_levene = stats.levene(*samples)
                levene_p = round(p_levene, 3)
                variance_status = "Homogeneous" if p_levene >= 0.05 else "Heterogeneous"
            except Exception as e:
                variance_status = f"Error: {str(e)}"
                levene_p = "Error"
        
        # --- Suggerimento test statistico ---
        if overall_normality == "Normal" and variance_status == "Homogeneous":
            suggested_test = "ANOVA + post-hoc Tukey"
        elif overall_normality == "Normal" and variance_status == "Heterogeneous":
            suggested_test = "Welch's ANOVA + post-hoc Games-Howell"
        else:
            suggested_test = "Kruskal-Wallis + post-hoc Dunn"

    result_row['Normality'] = overall_normality
    pvals_str = "; ".join([f"{grp}: {normality_pvalues[grp]}" for grp in groups_without_ctl])
    result_row['Shapiro p-values'] = pvals_str
    result_row['Variance Homogeneity'] = variance_status
    result_row['Levene p-value'] = levene_p
    result_row['Suggested Test'] = suggested_test
    result_row['Groups with sufficient data'] = ", ".join(sufficient_groups) if sufficient_groups else "None"
    
    cont_results.append(result_row)

# Visualizzazione dei risultati per le variabili continue
pd.DataFrame(cont_results)


,Variable,Type,Normality,Shapiro p-values,Variance Homogeneity,Levene p-value,Suggested Test,Groups with sufficient data
0,Age,Continuous,Normal,CTL: 0.339; DNV: 0.077; ADV: 0.895; DYS: 0.125,Homogeneous,0.853,ANOVA + post-hoc Tukey,"CTL, DNV, ADV, DYS"
1,Disease_Duration,Continuous,Normal,DNV: 0.93; ADV: 0.726; DYS: 0.06,Heterogeneous,0.002,Welch's ANOVA + post-hoc Games-Howell,"DNV, ADV, DYS"
2,LEDD,Continuous,Non-normal,DNV: 0.006; ADV: 0.528; DYS: 0.812,Heterogeneous,0.009,Kruskal-Wallis + post-hoc Dunn,"DNV, ADV, DYS"


In [52]:
import numpy as np

# ----------------------------
# PARTE 3: ANALISI VARIABILI ORDINALI (sempre non parametriche)
# ----------------------------
ord_results = []
for var in ordinal_vars:
    result_row = {'Variable': var, 'Type': 'Ordinal'}

    # Verifica gruppi con dati sufficienti (n ≥ 3)
    sufficient_groups = [
        grp for grp in groups
        if df[df['Group'] == grp][var].dropna().shape[0] >= 3
    ]

    result_row.update({
        'Normality': "Non-parametric test required",
        'Shapiro p-values': "Not applicable",
        'Variance Homogeneity': "Non-parametric test required",
        'Levene p-value': "Not applicable",
        'Suggested Test': "Kruskal-Wallis + post-hoc Dunn" if len(sufficient_groups) >= 2 else "Insufficient data for statistical testing",
        'Groups with sufficient data': ", ".join(sufficient_groups) if sufficient_groups else "None"
    })

    ord_results.append(result_row)

In [53]:
# ----------------------------
# PARTE 4: STATISTICHE DESCRITTIVE PER OGNI GRUPPO
# ----------------------------
def compute_descriptive_stats(df, var, group):
    data = df[df['Group'] == group][var].dropna()
    if data.empty:
        return {"n": 0, "Mean": np.nan, "SD": np.nan, "Median": np.nan, "Q1": np.nan, "Q3": np.nan}
    return {
        "n": len(data),
        "Mean": data.mean(),
        "SD": data.std(),
        "Median": data.median(),
        "Q1": data.quantile(0.25),
        "Q3": data.quantile(0.75)
    }

desc_stats_rows = []

# Statistiche per variabili continue
for var in continuous_vars:
    for group in groups:
        stats = compute_descriptive_stats(df, var, group)
        summary = f"{stats['Mean']:.1f} ± {stats['SD']:.1f}" if stats['n'] > 0 else "NA"
        desc_stats_rows.append({
            "Variable": var,
            "Group": group,
            "Type": "Continuous",
            "n": stats["n"],
            "Summary": summary
        })

# Statistiche per variabili ordinali
for var in ordinal_vars:
    for group in groups:
        stats = compute_descriptive_stats(df, var, group)
        summary = f"{stats['Median']:.1f} ({stats['Q1']:.1f}, {stats['Q3']:.1f})" if stats['n'] > 0 else "NA"
        desc_stats_rows.append({
            "Variable": var,
            "Group": group,
            "Type": "Ordinal",
            "n": stats["n"],
            "Summary": summary
        })

descriptive_stats_df = pd.DataFrame(desc_stats_rows)


In [54]:
# ----------------------------
# PARTE 5: VARIABILI CATEGORICHE PER GRUPPO
# ----------------------------
cat_results = []

for grp in groups:
    group_data = df[df['Group'] == grp]
    total = len(group_data)

    if 'Gender' in df.columns:
        n_F = sum(group_data['Gender'] == 'F')
        n_M = sum(group_data['Gender'] == 'M')
        cat_results.append({
            "Group": grp,
            "Variable": "Gender",
            "Type": "Categorical",
            "n": n_F + n_M,
            "Summary": f"{n_F} F / {n_M} M"
        })

    for var in ['DA_AGONIST_YES1NOT0', 'BDZ_YES1NOT0']:
        if grp == 'CTL':
            continue  # Escludi controlli per queste variabili
        if var in df.columns:
            values = group_data[var].dropna()
            n_yes = sum(values == 1)
            pct = round((n_yes / len(values)) * 100, 1) if len(values) > 0 else 0
            cat_results.append({
                "Group": grp,
                "Variable": var,
                "Type": "Categorical",
                "n": len(values),
                "Summary": f"{n_yes} ({pct}%)"
            })

categorical_summary_df = pd.DataFrame(cat_results)

# ----------------------------
# PARTE 6: TEST STATISTICI VARIABILI CATEGORICHE
# ----------------------------
cat_test_results = []

for var in categorical_vars:
    if var not in df.columns:
        continue

    temp_df = df.copy()
    temp_df[var] = temp_df[var].astype(str)

    # Escludi CTL tranne che per 'Gender'
    if var == 'Gender':
        groups_to_use = desired_order
    else:
        groups_to_use = [g for g in desired_order if g != 'CTL']

    temp_df = temp_df[temp_df['Group'].isin(groups_to_use)]

    try:
        contingency = pd.crosstab(temp_df['Group'], temp_df[var])
        existing_rows = [g for g in desired_order if g in contingency.index]
        contingency = contingency.loc[existing_rows]

        # Gruppi con almeno n≥3 osservazioni
        sufficient_groups = [g for g in existing_rows if contingency.loc[g].sum() >= 3]

        if len(sufficient_groups) < 2:
            cat_test_results.append({
                "Variable": var,
                "Type": "Categorical",
                "Test Used": "None",
                "Statistic": "NA",
                "p-value": "NA",
                "Note": "Insufficient groups with n≥3"
            })
        elif contingency.shape[0] > 1 and contingency.shape[1] > 1:
            filtered = contingency.loc[sufficient_groups]
            from scipy.stats import chi2_contingency  # assicura import corretto
            chi2, p_val, dof, expected = chi2_contingency(filtered)
            expected_min = expected.min()
            pct_under_5 = (expected < 5).sum() / expected.size * 100
            test_name = "Chi-square"
            note = ""
            if pct_under_5 > 20 or expected_min < 1:
                test_name += " (warning: low expected counts)"
                note = f"{pct_under_5:.1f}% cells with expected count < 5"

            cat_test_results.append({
                "Variable": var,
                "Type": "Categorical",
                "Test Used": test_name,
                "Statistic": f"chi2={chi2:.3f}, dof={dof}",
                "p-value": round(p_val, 3),
                "Note": note,
                "Groups included": ", ".join(sufficient_groups)
            })
        else:
            cat_test_results.append({
                "Variable": var,
                "Type": "Categorical",
                "Test Used": "None",
                "Statistic": "NA",
                "p-value": "NA",
                "Note": f"Insufficient data: table shape {contingency.shape}"
            })
    except Exception as e:
        cat_test_results.append({
            "Variable": var,
            "Type": "Categorical",
            "Test Used": "Error",
            "Statistic": "Error",
            "p-value": "NA",
            "Note": f"Error in crosstab: {str(e)}"
        })

categorical_tests_df = pd.DataFrame(cat_test_results)

# ----------------------------
# UNIONE RISULTATI
# ----------------------------
numeric_summary_df = pd.DataFrame(cont_results + ord_results)
numeric_summary_df = numeric_summary_df[[ 
    'Variable', 'Type', 'Shapiro p-values', 'Normality',
    'Levene p-value', 'Variance Homogeneity',
    'Suggested Test', 'Groups with sufficient data'
]]

# Descrittive: combina variabili numeriche + variabili categoriche (senza CTL per BDZ/DA)
descriptive_df = pd.concat([descriptive_stats_df, categorical_summary_df])


In [56]:
import os
import pandas as pd

# Percorso del file di output
output_file = os.path.join(
    r"C:\Users\Lorenzo\Desktop\scoring/demographics/", 
    "statistical_analysis_summary.xlsx"
)

# Salvataggio dei DataFrame nei rispettivi fogli del file Excel
with pd.ExcelWriter(output_file) as writer:
    numeric_summary_df.to_excel(writer, sheet_name="Statistical Tests Summary", index=False)
    descriptive_df.to_excel(writer, sheet_name="Descriptive Statistics", index=False)
    categorical_tests_df.to_excel(writer, sheet_name="Categorical Tests", index=False)

# Messaggio di conferma
print(f"Riepilogo completo dell'analisi statistica salvato in: {output_file}")


Riepilogo completo dell'analisi statistica salvato in: C:\Users\Lorenzo\Desktop\scoring/demographics/statistical_analysis_summary.xlsx


In [24]:
import pandas as pd
import numpy as np
import os
import scipy.stats as stats
import warnings
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import pingouin as pg

# ============================
# FUNZIONI AUSILIARIE
# ============================

def pairwise_dunn(data, var, groups_used, desired_order, group_col='Group', alpha=0.05):
    sub_df = data[data[group_col].isin(groups_used)][[var, group_col]].dropna()
    if len(sub_df) < 2 or len(groups_used) < 2:
        return "Insufficient data"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning, module="scikit_posthocs")
        try:
            dunn_res = sp.posthoc_dunn(sub_df, val_col=var, group_col=group_col, p_adjust='bonferroni')
        except Exception as e:
            return f"Error in Dunn test: {e}"
    sig = []
    groups_sorted = sorted(groups_used, key=lambda x: desired_order.index(x) if x in desired_order else len(desired_order))
    for i in range(len(groups_sorted)):
        for j in range(i+1, len(groups_sorted)):
            p_val = dunn_res.loc[groups_sorted[i], groups_sorted[j]]
            if p_val < alpha:
                p_str = np.format_float_scientific(p_val, precision=3)
                sig.append(f"{groups_sorted[i]} vs {groups_sorted[j]} (p={p_str})")
    return "; ".join(sig) if sig else "None"

def welch_anova(samples, epsilon=1e-6):
    k = len(samples)
    if k < 2 or any(len(s) < 2 for s in samples):
        return np.nan, np.nan
    ns = np.array([len(s) for s in samples])
    means = np.array([np.mean(s) for s in samples])
    variances = np.array([np.var(s, ddof=1) for s in samples])
    variances = np.where(variances == 0, epsilon, variances)
    weights = ns / variances
    grand_mean = np.sum(weights * means) / np.sum(weights)
    numerator = np.sum(weights * (means - grand_mean)**2)
    F = numerator / (k - 1)
    denominator_terms = (1 - weights/np.sum(weights))**2 / (ns - 1)
    if np.sum(denominator_terms) == 0:
        return np.nan, np.nan
    df_den = (k**2 - 1) / (3 * np.sum(denominator_terms))
    if not np.isfinite(F) or not np.isfinite(df_den) or df_den <= 0:
        return np.nan, np.nan
    p_val = 1 - stats.f.cdf(F, k - 1, df_den)
    return F, p_val

def group_descriptive_stats_dict(data, groups, var, var_type, norm_status):
    d = {}
    for grp in groups:
        vals = data[data['Group'] == grp][var].dropna()
        if len(vals) == 0:
            d[grp] = "-"
        elif len(vals) < 3:
            if np.std(vals, ddof=1) == 0:
                d[grp] = f"{np.round(np.mean(vals),1)}"
            else:
                d[grp] = f"{len(vals)} values"
        else:
            if var_type == "Continuous" and norm_status == "Normal":
                d[grp] = f"{np.round(np.mean(vals),1)} ± {np.round(np.std(vals, ddof=1),1)}"
            else:
                median_val = np.median(vals)
                q1 = np.percentile(vals, 25)
                q3 = np.percentile(vals, 75)
                d[grp] = f"{np.round(median_val,1)} ({np.round(q1,1)}-{np.round(q3,1)})"
    return d

# ============================
# CARICAMENTO DEI DATI
# ============================

data_dir = r"C:\Users\Lorenzo\Desktop\scoring/demographics"
data_file = os.path.join(data_dir, "summary_extracted.xlsx")
stats_file = os.path.join(data_dir, "statistical_analysis_summary.xlsx")
output_file = os.path.join(data_dir, "final_analysis_with_tests.xlsx")

if not os.path.exists(data_file):
    print(f"ERRORE: File dati non trovato: {data_file}")
    exit(1)
if not os.path.exists(stats_file):
    print(f"ERRORE: File analisi statistica non trovato: {stats_file}")
    exit(1)

try:
    df = pd.read_excel(data_file)
    df['Group'] = df['Group'].astype('category')
    stats_summary_df = pd.read_excel(stats_file, sheet_name="Statistical Tests Summary")
except Exception as e:
    print(f"ERRORE nel caricamento dei file: {e}")
    exit(1)

# ============================
# VARIABILI DA ANALIZZARE
# ============================

continuous_vars = ['Age', 'Disease_Duration', 'LEDD', 'UPDRSIV4.1']
ordinal_vars = ['UPDRSI', 'UPDRSII', 'UPDRSIII', 'UPDRSIV', 'AIMs', 'Hoen_Year']
categorical_vars = ['Gender', 'DA_AGONIST_YES1NOT0', 'BDZ_YES1NOT0']

all_vars = continuous_vars + ordinal_vars + categorical_vars
existing_vars = [var for var in all_vars if var in df.columns]
if len(existing_vars) < len(all_vars):
    missing_vars = set(all_vars) - set(existing_vars)
    print(f"ATTENZIONE: Alcune variabili non sono presenti nel dataframe: {missing_vars}")
    continuous_vars = [var for var in continuous_vars if var in existing_vars]
    ordinal_vars = [var for var in ordinal_vars if var in existing_vars]
    categorical_vars = [var for var in categorical_vars if var in existing_vars]

desired_order = ['CTL', 'DNV', 'ADV', 'DYS']
available_groups = [grp for grp in desired_order if grp in df['Group'].unique()]

# ============================
# DESCRIPTIVE STATISTICS
# ============================

desc_rows = []

# Variabili continue
for var in continuous_vars:
    row = {"Variable": var, "Type": "Continuous"}
    try:
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        norm_status = stats_row["Normality"]
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        sufficient_groups = [] if sufficient_groups_str == "None" else [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception:
        norm_status = "Unknown"
        sufficient_groups = []

    row["Overall Normality"] = norm_status
    stats_dict = group_descriptive_stats_dict(df, available_groups, var, "Continuous", norm_status)
    for grp in available_groups:
        row[grp] = stats_dict.get(grp, "-")
    desc_rows.append(row)

# Variabili ordinali
for var in ordinal_vars:
    row = {"Variable": var, "Type": "Ordinal"}
    try:
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        sufficient_groups = [] if sufficient_groups_str == "None" else [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception:
        sufficient_groups = []

    row["Overall Normality"] = "Non-parametric"
    stats_dict = group_descriptive_stats_dict(df, available_groups, var, "Ordinal", "Non-normal")
    for grp in available_groups:
        row[grp] = stats_dict.get(grp, "-")
    desc_rows.append(row)

descriptive_df = pd.DataFrame(desc_rows)

# ============================
# SALVATAGGIO RISULTATI
# ============================

output_excel = os.path.join(data_dir, "statistical_analysis_summary.xlsx")

with pd.ExcelWriter(output_excel) as writer:
    # Questi DataFrame devono essere già definiti nel tuo notebook:
    numeric_summary_df.to_excel(writer, sheet_name="Statistical Tests Summary", index=False)
    descriptive_df.to_excel(writer, sheet_name="Descriptive Statistics", index=False)
    categorical_tests_df.to_excel(writer, sheet_name="Categorical Tests", index=False)

print(f"Riepilogo completo dell'analisi statistica salvato in: {output_excel}")


Riepilogo completo dell'analisi statistica salvato in: C:\Users\Lorenzo\Desktop\scoring/demographics\statistical_analysis_summary.xlsx


In [25]:
# ============================
# 2. Creazione della tabella Test Statistics
# ============================

test_rows = []

# Funzione per verificare se un gruppo ha dati sufficienti
def has_sufficient_data(data, var, group, min_size=3):
    vals = data[data['Group'] == group][var].dropna()
    return len(vals) >= min_size

# Prima per le variabili continue
for var in continuous_vars:
    res = {"Variable": var, "Type": "Continuous"}
    try:
        # Cerca le informazioni nel file summary
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        norm_status = stats_row["Normality"]
        var_status = stats_row["Variance Homogeneity"]
        suggested_test = stats_row["Suggested Test"]
        
        # Ottieni quali gruppi hanno dati sufficienti
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        if sufficient_groups_str == "None" or sufficient_groups_str.strip() == "":
            sufficient_groups = []
        else:
            sufficient_groups = [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception as e:
        print(f"Errore nel recupero delle statistiche per {var}: {e}")
        norm_status = "Unknown"
        var_status = "Unknown"
        suggested_test = "Unknown"
        sufficient_groups = []
    
    res["Normality"] = norm_status
    res["Variance"] = var_status
    res["Groups Used"] = ", ".join(sufficient_groups)
    
    # Se non ci sono abbastanza gruppi con dati sufficienti, salta i test
    if len(sufficient_groups) < 2:
        res["Test Used"] = "Insufficient data"
        res["Statistic"] = "NA"
        res["p-value"] = "NA"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Prepara i campioni per i test
    samples = []
    for grp in sufficient_groups:
        s = df[df['Group'] == grp][var].dropna().values
        samples.append(s)
    
    # Esegui il test appropriato in base alle caratteristiche dei dati
    if norm_status == "Normal" and var_status == "Homogeneous":
        try:
            f_stat, p_val = stats.f_oneway(*samples)
            res["Test Used"] = "One-Way ANOVA"
            res["Statistic"] = np.round(f_stat, 3)
            res["p-value"] = np.format_float_scientific(p_val, precision=3)
        except Exception as e:
            res["Test Used"] = "One-Way ANOVA"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            test_rows.append(res)
            continue
        
        # Post-hoc test se ANOVA è significativa
        if p_val < 0.05:
            try:
                tukey_data = df[df["Group"].isin(sufficient_groups)][[var, "Group"]].dropna()
                tukey = pairwise_tukeyhsd(endog=tukey_data[var],
                                          groups=tukey_data["Group"],
                                          alpha=0.05)
                sig_comparisons = []
                try:
                    # Creare il test di Tukey manualmente per ottenere i p-value
                    from statsmodels.stats.multicomp import MultiComparison
                    mc = MultiComparison(tukey_data[var], tukey_data["Group"])
                    tukey_result = mc.tukeyhsd().summary()
                    pvals = mc.pvalues
                    
                    # Estrai le coppie ordinate
                    pairs = []
                    for i, grp1 in enumerate(mc.groupsunique):
                        for j, grp2 in enumerate(mc.groupsunique):
                            if j > i:  # evita duplicati e confronti con se stesso
                                pairs.append((grp1, grp2))
                    
                    # Abbina ogni coppia al suo p-value
                    for k, (grp1, grp2) in enumerate(pairs):
                        p_val = pvals[k]
                        if p_val < 0.05:
                            p_str = np.format_float_scientific(p_val, precision=3)
                            sig_comparisons.append(f"{grp1} vs {grp2} (p={p_str})")
                    
                    res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None"
                except Exception as e:
                    # Fallback al metodo originale se non funziona
                    for row in tukey.summary().data[1:]:
                        group1, group2, meandiff, lower, upper, reject = row
                        if reject:
                            sig_comparisons.append(f"{group1} vs {group2}")
                    res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None (Error getting p-values)"
            except Exception as e:
                res["Post-hoc"] = f"Error in Tukey: {e}"
        else:
            res["Post-hoc"] = "Not significant"
    
    elif norm_status == "Normal" and var_status == "Heterogeneous":
        try:
            f_stat, p_val = welch_anova(samples)
            res["Test Used"] = "Welch's ANOVA"
            res["Statistic"] = f"F={np.round(f_stat,3)}" if np.isfinite(f_stat) else "NA"
            res["p-value"] = np.format_float_scientific(p_val, precision=3) if np.isfinite(p_val) else "NA"
        except Exception as e:
            res["Test Used"] = "Welch's ANOVA"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            res["Post-hoc"] = "NA"
            test_rows.append(res)
            continue
        
        # Post-hoc test se Welch's ANOVA è significativa
        if np.isfinite(p_val) and p_val < 0.05:
            try:
                # Post hoc con Games-Howell per Welch's ANOVA
                gh = pg.pairwise_gameshowell(dv=var, between="Group", 
                                            data=df[df["Group"].isin(sufficient_groups)].dropna(subset=[var]))
                sig_comparisons = []
                for _, row in gh.iterrows():
                    if row['pval'] < 0.05:
                        p_str = np.format_float_scientific(row['pval'], precision=3)
                        sig_comparisons.append(f"{row['A']} vs {row['B']} (p={p_str})")
                res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None"
            except Exception as e:
                res["Post-hoc"] = f"Error in post-hoc Games-Howell: {e}"
        else:
            res["Post-hoc"] = "Not significant"
    
    else:
        # Per i casi non normali o quando non è possibile verificare la normalità
        try:
            h_stat, p_val = stats.kruskal(*samples)
            res["Test Used"] = "Kruskal-Wallis"
            res["Statistic"] = np.round(h_stat, 3)
            res["p-value"] = np.format_float_scientific(p_val, precision=3)
        except Exception as e:
            res["Test Used"] = "Kruskal-Wallis"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            res["Post-hoc"] = "NA"
            test_rows.append(res)
            continue
        
        # Post-hoc test se Kruskal-Wallis è significativo
        if p_val < 0.05:
            try:
                dunn_res = pairwise_dunn(df, var, groups_used=sufficient_groups, 
                                        desired_order=desired_order, group_col="Group", alpha=0.05)
                res["Post-hoc"] = dunn_res
            except Exception as e:
                res["Post-hoc"] = f"Error in Dunn: {e}"
        else:
            res["Post-hoc"] = "Not significant"
    
    test_rows.append(res)

# Poi per le variabili ordinali (sempre Kruskal-Wallis)
for var in ordinal_vars:
    res = {"Variable": var, "Type": "Ordinal"}
    try:
        # Cerca le informazioni nel file summary
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        # Ottieni quali gruppi hanno dati sufficienti
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        if sufficient_groups_str == "None" or sufficient_groups_str.strip() == "":
            sufficient_groups = []
        else:
            sufficient_groups = [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception:
        sufficient_groups = []
    
    res["Normality"] = "Non-parametric"
    res["Variance"] = "Non-parametric"
    res["Groups Used"] = ", ".join(sufficient_groups)
    
    # Se non ci sono abbastanza gruppi con dati sufficienti, salta i test
    if len(sufficient_groups) < 2:
        res["Test Used"] = "Insufficient data"
        res["Statistic"] = "NA"
        res["p-value"] = "NA"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Prepara i campioni per il test Kruskal-Wallis
    samples = []
    for grp in sufficient_groups:
        s = df[df['Group'] == grp][var].dropna().values
        samples.append(s)
    
    # Esegui Kruskal-Wallis (sempre per variabili ordinali)
    try:
        h_stat, p_val = stats.kruskal(*samples)
        res["Test Used"] = "Kruskal-Wallis"
        res["Statistic"] = np.round(h_stat, 3)
        res["p-value"] = np.format_float_scientific(p_val, precision=3)
    except Exception as e:
        res["Test Used"] = "Kruskal-Wallis"
        res["Statistic"] = f"Error: {e}"
        res["p-value"] = "Error"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Post-hoc test se Kruskal-Wallis è significativo
    if p_val < 0.05:
        try:
            dunn_res = pairwise_dunn(df, var, groups_used=sufficient_groups, 
                                    desired_order=desired_order, group_col="Group", alpha=0.05)
            res["Post-hoc"] = dunn_res
        except Exception as e:
            res["Post-hoc"] = f"Error in Dunn: {e}"
    else:
        res["Post-hoc"] = "Not significant"
    
    test_rows.append(res)

# Crea il dataframe finale
test_df = pd.DataFrame(test_rows)


In [26]:
# ============================
# 2. Creazione della tabella Test Statistics
# ============================

test_rows = []

# Funzione per verificare se un gruppo ha dati sufficienti
def has_sufficient_data(data, var, group, min_size=3):
    vals = data[data['Group'] == group][var].dropna()
    return len(vals) >= min_size

# Prima per le variabili continue
for var in continuous_vars:
    res = {"Variable": var, "Type": "Continuous"}
    try:
        # Cerca le informazioni nel file summary
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        norm_status = stats_row["Normality"]
        var_status = stats_row["Variance Homogeneity"]
        suggested_test = stats_row["Suggested Test"]
        
        # Ottieni quali gruppi hanno dati sufficienti
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        if sufficient_groups_str == "None" or sufficient_groups_str.strip() == "":
            sufficient_groups = []
        else:
            sufficient_groups = [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception as e:
        print(f"Errore nel recupero delle statistiche per {var}: {e}")
        norm_status = "Unknown"
        var_status = "Unknown"
        suggested_test = "Unknown"
        sufficient_groups = []
    
    res["Normality"] = norm_status
    res["Variance"] = var_status
    res["Groups Used"] = ", ".join(sufficient_groups)
    
    # Se non ci sono abbastanza gruppi con dati sufficienti, salta i test
    if len(sufficient_groups) < 2:
        res["Test Used"] = "Insufficient data"
        res["Statistic"] = "NA"
        res["p-value"] = "NA"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Prepara i campioni per i test
    samples = []
    for grp in sufficient_groups:
        s = df[df['Group'] == grp][var].dropna().values
        samples.append(s)
    
    # Esegui il test appropriato in base alle caratteristiche dei dati
    if norm_status == "Normal" and var_status == "Homogeneous":
        try:
            f_stat, p_val = stats.f_oneway(*samples)
            res["Test Used"] = "One-Way ANOVA"
            res["Statistic"] = np.round(f_stat, 3)
            res["p-value"] = np.format_float_scientific(p_val, precision=3)
        except Exception as e:
            res["Test Used"] = "One-Way ANOVA"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            test_rows.append(res)
            continue
        
        # Post-hoc test se ANOVA è significativa
        if p_val < 0.05:
            try:
                tukey_data = df[df["Group"].isin(sufficient_groups)][[var, "Group"]].dropna()
                tukey = pairwise_tukeyhsd(endog=tukey_data[var],
                                        groups=tukey_data["Group"],
                                        alpha=0.05)
                sig_comparisons = []
                # Estrai solo le comparazioni significative con p-value
                try:
                    # Creare il test di Tukey manualmente per ottenere i p-value
                    from statsmodels.stats.multicomp import MultiComparison
                    mc = MultiComparison(tukey_data[var], tukey_data["Group"])
                    tukey_result = mc.tukeyhsd().summary()
                    pvals = mc.pvalues
                    
                    # Estrai le coppie ordinate
                    pairs = []
                    for i, grp1 in enumerate(mc.groupsunique):
                        for j, grp2 in enumerate(mc.groupsunique):
                            if j > i:  # evita duplicati e confronti con se stesso
                                pairs.append((grp1, grp2))
                    
                    # Abbina ogni coppia al suo p-value
                    for k, (grp1, grp2) in enumerate(pairs):
                        p_val = pvals[k]
                        if p_val < 0.05:
                            p_str = np.format_float_scientific(p_val, precision=3)
                            sig_comparisons.append(f"{grp1} vs {grp2} (p={p_str})")
                    
                    res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None"
                except Exception as e:
                    # Fallback al metodo originale se non funziona
                    for row in tukey.summary().data[1:]:
                        group1, group2, meandiff, lower, upper, reject = row
                        if reject:
                            sig_comparisons.append(f"{group1} vs {group2}")
                    res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None (Error getting p-values)"
            except Exception as e:
                res["Post-hoc"] = f"Error in Tukey: {e}"
        else:
            res["Post-hoc"] = "Not significant"
        
    elif norm_status == "Normal" and var_status == "Heterogeneous":
        try:
            f_stat, p_val = welch_anova(samples)
            res["Test Used"] = "Welch's ANOVA"
            res["Statistic"] = f"F={np.round(f_stat,3)}" if np.isfinite(f_stat) else "NA"
            res["p-value"] = np.format_float_scientific(p_val, precision=3) if np.isfinite(p_val) else "NA"
        except Exception as e:
            res["Test Used"] = "Welch's ANOVA"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            res["Post-hoc"] = "NA"
            test_rows.append(res)
            continue
        
        # Post-hoc test se Welch's ANOVA è significativa
        if np.isfinite(p_val) and p_val < 0.05:
            try:
                # Post hoc con Games-Howell per Welch's ANOVA
                gh = pg.pairwise_gameshowell(dv=var, between="Group", 
                                            data=df[df["Group"].isin(sufficient_groups)].dropna(subset=[var]))
                sig_comparisons = []
                for _, row in gh.iterrows():
                    if row['pval'] < 0.05:
                        p_str = np.format_float_scientific(row['pval'], precision=3)
                        sig_comparisons.append(f"{row['A']} vs {row['B']} (p={p_str})")
                res["Post-hoc"] = "; ".join(sig_comparisons) if sig_comparisons else "None"
            except Exception as e:
                res["Post-hoc"] = f"Error in post-hoc Games-Howell: {e}"
        else:
            res["Post-hoc"] = "Not significant"
    
    else:
        # Per i casi non normali o quando non è possibile verificare la normalità
        try:
            h_stat, p_val = stats.kruskal(*samples)
            res["Test Used"] = "Kruskal-Wallis"
            res["Statistic"] = np.round(h_stat, 3)
            res["p-value"] = np.format_float_scientific(p_val, precision=3)
        except Exception as e:
            res["Test Used"] = "Kruskal-Wallis"
            res["Statistic"] = f"Error: {e}"
            res["p-value"] = "Error"
            res["Post-hoc"] = "NA"
            test_rows.append(res)
            continue
        
        # Post-hoc test se Kruskal-Wallis è significativo
        if p_val < 0.05:
            try:
                dunn_res = pairwise_dunn(df, var, groups_used=sufficient_groups, 
                                        desired_order=desired_order, group_col="Group", alpha=0.05)
                res["Post-hoc"] = dunn_res
            except Exception as e:
                res["Post-hoc"] = f"Error in Dunn: {e}"
        else:
            res["Post-hoc"] = "Not significant"
    
    test_rows.append(res)

# Poi per le variabili ordinali (sempre Kruskal-Wallis)
for var in ordinal_vars:
    res = {"Variable": var, "Type": "Ordinal"}
    try:
        # Cerca le informazioni nel file summary
        stats_row = stats_summary_df[stats_summary_df['Variable'] == var].iloc[0]
        # Ottieni quali gruppi hanno dati sufficienti
        sufficient_groups_str = stats_row["Groups with sufficient data"]
        if sufficient_groups_str == "None" or sufficient_groups_str.strip() == "":
            sufficient_groups = []
        else:
            sufficient_groups = [g.strip() for g in sufficient_groups_str.split(",")]
    except Exception:
        sufficient_groups = []
    
    res["Normality"] = "Non-parametric"
    res["Variance"] = "Non-parametric"
    res["Groups Used"] = ", ".join(sufficient_groups)
    
    # Se non ci sono abbastanza gruppi con dati sufficienti, salta i test
    if len(sufficient_groups) < 2:
        res["Test Used"] = "Insufficient data"
        res["Statistic"] = "NA"
        res["p-value"] = "NA"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Prepara i campioni per il test Kruskal-Wallis
    samples = []
    for grp in sufficient_groups:
        s = df[df['Group'] == grp][var].dropna().values
        samples.append(s)
    
    # Esegui Kruskal-Wallis (sempre per variabili ordinali)
    try:
        h_stat, p_val = stats.kruskal(*samples)
        res["Test Used"] = "Kruskal-Wallis"
        res["Statistic"] = np.round(h_stat, 3)
        res["p-value"] = np.format_float_scientific(p_val, precision=3)
    except Exception as e:
        res["Test Used"] = "Kruskal-Wallis"
        res["Statistic"] = f"Error: {e}"
        res["p-value"] = "Error"
        res["Post-hoc"] = "NA"
        test_rows.append(res)
        continue
    
    # Post-hoc test se Kruskal-Wallis è significativo
    if p_val < 0.05:
        try:
            dunn_res = pairwise_dunn(df, var, groups_used=sufficient_groups, 
                                    desired_order=desired_order, group_col="Group", alpha=0.05)
            res["Post-hoc"] = dunn_res
        except Exception as e:
            res["Post-hoc"] = f"Error in Dunn: {e}"
    else:
        res["Post-hoc"] = "Not significant"
    
    test_rows.append(res)

# Creazione del DataFrame finale con i risultati
test_df = pd.DataFrame(test_rows)


In [29]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Percorsi ai file
demographics_path = r"C:\Users\Lorenzo\Desktop\scoring\demographics/summary_extracted.xlsx"
sleep_path = r"C:\Users\Lorenzo\Desktop\scoring\sleep_characteristics.xlsx"

# Cartella di destinazione per i grafici di correlazione
correlation_folder = r"C:\Users\Lorenzo\Desktop\scoring\correlation"
os.makedirs(correlation_folder, exist_ok=True)

# Carica i dati
df_dem = pd.read_excel(demographics_path)
df_sleep = pd.read_excel(sleep_path)

# Unione dei DataFrame: usa "ID" per i demografici e "Patient" per lo scoring
df_merged = pd.merge(df_dem, df_sleep, left_on="ID", right_on="Patient")

# Escludi i controlli (gruppo "CTL")
df_merged = df_merged[df_merged["Group_x"] != "CTL"]

# Calcolo della correlazione di Pearson utilizzando solo i dati dei pazienti (non controlli)
r, p_value = pearsonr(df_merged["Disease_Duration"], df_merged["Total Sleep Time (min)"])
print(f"Correlazione Pearson: r = {r:.3f}, p-value = {p_value:.3f}")

# Creazione del grafico di dispersione con linea di regressione
plt.figure(figsize=(8, 6))
sns.regplot(x="Disease_Duration", y="Total Sleep Time (min)", data=df_merged, ci=95)
plt.title("All Patients")
plt.xlabel("Disease Duration")
plt.ylabel("Total Sleep Time (min)")
plt.grid(True, linestyle="--", alpha=0.7)

# Aggiungi l'annotazione dei risultati di Pearson nell'angolo in alto a destra
plt.text(0.95, 0.95, f"r = {r:.3f}\\np = {p_value:.3f}", 
         transform=plt.gca().transAxes, 
         horizontalalignment='right', 
         verticalalignment='top',
         fontsize=12)

# Salvataggio del grafico nella cartella "correlation"
output_path = os.path.join(correlation_folder, "disease_duration_vs_tst_no_CTL.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Grafico salvato in: {output_path}")

# Lista dei gruppi da analizzare
groups = ["DNV", "ADV", "DYS"]

for group in groups:
    # Filtra il DataFrame per il gruppo corrente
    df_group = df_merged[df_merged["Group_x"] == group]
    
    # Calcola la correlazione di Pearson per il gruppo corrente
    r, p_value = pearsonr(df_group["Disease_Duration"], df_group["Total Sleep Time (min)"])
    print(f"Gruppo {group} - Correlazione Pearson: r = {r:.3f}, p-value = {p_value:.3f}")
    
    # Crea il grafico di dispersione con linea di regressione
    plt.figure(figsize=(8, 6))
    sns.regplot(x="Disease_Duration", y="Total Sleep Time (min)", data=df_group, ci=95)
    plt.title(f"{group}")
    plt.xlabel("Disease Duration")
    plt.ylabel("Total Sleep Time (min)")
    plt.grid(True, linestyle="--", alpha=0.7)
    
    # Aggiungi l'annotazione dei risultati di Pearson in alto a destra
    plt.text(0.95, 0.95, f"r = {r:.3f}\\np = {p_value:.3f}", 
             transform=plt.gca().transAxes, 
             horizontalalignment='right', 
             verticalalignment='top', 
             fontsize=12,
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    
    # Salva il grafico per il gruppo corrente
    output_path = os.path.join(correlation_folder, f"disease_duration_vs_tst_{group}.png")
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"Grafico salvato in: {output_path}")


Correlazione Pearson: r = -0.472, p-value = 0.023
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_tst_no_CTL.png
Gruppo DNV - Correlazione Pearson: r = 0.730, p-value = 0.161
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_tst_DNV.png
Gruppo ADV - Correlazione Pearson: r = -0.696, p-value = 0.055
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_tst_ADV.png
Gruppo DYS - Correlazione Pearson: r = -0.324, p-value = 0.362
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_tst_DYS.png


In [31]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Percorsi ai file
demographics_path = r"C:\Users\Lorenzo\Desktop\scoring\demographics/summary_extracted.xlsx"
sleep_path = r"C:\Users\Lorenzo\Desktop\scoring\sleep_characteristics.xlsx"

# Cartella di destinazione per i grafici di correlazione
correlation_folder = r"C:\Users\Lorenzo\Desktop\scoring\correlation"
os.makedirs(correlation_folder, exist_ok=True)

# Carica i dati
df_dem = pd.read_excel(demographics_path)
df_sleep = pd.read_excel(sleep_path)

# Unione dei DataFrame: "ID" per i demografici e "Patient" per lo scoring
df_merged = pd.merge(df_dem, df_sleep, left_on="ID", right_on="Patient")

# Escludi i controlli (gruppo "CTL")
df_merged = df_merged[df_merged["Group_x"] != "CTL"]

# ANALISI: Sleep Efficiency (%) vs Disease Duration

# 1. Analisi per tutti i pazienti (escludendo CTL) utilizzando Spearman
r_all, p_all = spearmanr(df_merged["Disease_Duration"], df_merged["Sleep Efficiency (%)"])
print(f"All patients - Spearman correlation: r = {r_all:.3f}, p = {p_all:.3f}")

plt.figure(figsize=(8, 6))
sns.regplot(x="Disease_Duration", y="Sleep Efficiency (%)", data=df_merged, ci=95)
plt.title("All Patients (non-CTL)")
plt.xlabel("Disease Duration")
plt.ylabel("Sleep Efficiency (%)")
plt.grid(True, linestyle="--", alpha=0.7)
plt.text(0.95, 0.95, f"r = {r_all:.3f}\\np = {p_all:.3f}", 
         transform=plt.gca().transAxes, 
         horizontalalignment='right', 
         verticalalignment='top', 
         fontsize=12,
         bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
output_path_all = os.path.join(correlation_folder, "disease_duration_vs_sleep_efficiency_no_CTL_all.png")
plt.savefig(output_path_all, dpi=300, bbox_inches="tight")
plt.close()
print(f"Grafico salvato in: {output_path_all}")

# 2. Analisi per ciascun gruppo: DNV, ADV, DYS
groups = ["DNV", "ADV", "DYS"]

for group in groups:
    df_group = df_merged[df_merged["Group_x"] == group]
    r_group, p_group = spearmanr(df_group["Disease_Duration"], df_group["Sleep Efficiency (%)"])
    print(f"Gruppo {group} - Spearman correlation: r = {r_group:.3f}, p = {p_group:.3f}")
    
    plt.figure(figsize=(8, 6))
    sns.regplot(x="Disease_Duration", y="Sleep Efficiency (%)", data=df_group, ci=95)
    plt.title(f"Group: {group}")
    plt.xlabel("Disease Duration")
    plt.ylabel("Sleep Efficiency (%)")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.text(0.95, 0.95, f"r = {r_group:.3f}\\np = {p_group:.3f}", 
             transform=plt.gca().transAxes, 
             horizontalalignment='right', 
             verticalalignment='top', 
             fontsize=12,
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    output_path_group = os.path.join(correlation_folder, f"disease_duration_vs_sleep_efficiency_{group}.png")
    plt.savefig(output_path_group, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Grafico salvato in: {output_path_group}")


All patients - Spearman correlation: r = -0.360, p = 0.092
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_sleep_efficiency_no_CTL_all.png
Gruppo DNV - Spearman correlation: r = 0.100, p = 0.873
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_sleep_efficiency_DNV.png
Gruppo ADV - Spearman correlation: r = -0.119, p = 0.779
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_sleep_efficiency_ADV.png
Gruppo DYS - Spearman correlation: r = -0.158, p = 0.663
Grafico salvato in: C:\Users\Lorenzo\Desktop\scoring\correlation\disease_duration_vs_sleep_efficiency_DYS.png


In [32]:
# Step 1: Lettura delle Colonne del Dataset
import pandas as pd

# Caricamento dei dati dei pazienti
percorso_file = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/pazienti_parkinson_filtrati.xlsx"
df = pd.read_excel(percorso_file)

print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")

# Mostra l'elenco completo delle colonne
print("\nElenco delle colonne nel dataset:")
for i, colonna in enumerate(df.columns):
    print(f"{i+1}. {colonna}")

# Mostra le prime righe per vedere i dati
print("\nPrime 5 righe del dataset:")
display(df.head())

# Verifica i gruppi presenti (se esiste una colonna Group)
gruppi_colonne = [col for col in df.columns if 'Group' in col or 'group' in col.lower()]
if gruppi_colonne:
    for col in gruppi_colonne:
        print(f"\nDistribuzione della colonna {col}:")
        display(df[col].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: '/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/pazienti_parkinson_filtrati.xlsx'

In [ ]:
# Step 2: Analisi della Sleep Efficiency con Salvataggio Figure
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot
import os

# Impostazioni per visualizzazione dei grafici
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# Creazione della cartella per salvare le figure
output_dir = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/scoring/confounding"
os.makedirs(output_dir, exist_ok=True)
print(f"Le figure saranno salvate in: {output_dir}")

# Funzione per salvare figura
def save_figure(name):
    plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=300, bbox_inches='tight')
    plt.close()

# Caricamento del dataset
percorso_file = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/pazienti_parkinson_filtrati.xlsx"
df = pd.read_excel(percorso_file)
print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")

# Variabili
var_dipendente = 'Sleep Efficiency (%)'
var_gruppo = 'Group'

# Statistiche descrittive
print(f"\nStatistiche descrittive per {var_dipendente} per gruppo:")
stats_by_group = df.groupby(var_gruppo)[var_dipendente].describe()
print(stats_by_group)

# Boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x=var_gruppo, y=var_dipendente)
plt.title(f'{var_dipendente} per Gruppo')
plt.ylabel(var_dipendente)
plt.xlabel('Gruppo')
plt.tight_layout()
save_figure("01_boxplot_sleep_efficiency_by_group")
print("Figura 1: Boxplot di Sleep Efficiency per gruppo salvata")

# Test di normalità
print("\nTest di normalità per gruppo (Shapiro-Wilk):")
gruppi_dati = []
gruppi_nomi = []
normalita_ok = True
for gruppo in df[var_gruppo].unique():
    dati = df[df[var_gruppo] == gruppo][var_dipendente].dropna()
    if len(dati) > 3:
        gruppi_dati.append(dati)
        gruppi_nomi.append(gruppo)
        stat, p = stats.shapiro(dati)
        normalita_gruppo = p > 0.05
        normalita_ok = normalita_ok and normalita_gruppo
        print(f"Gruppo {gruppo}: statistica = {stat:.4f}, p-value = {p:.4f}, normalità = {'Sì' if normalita_gruppo else 'No'}")

print(f"\nTutti i gruppi seguono distribuzione normale: {'Sì' if normalita_ok else 'No'}")
print(f"Approccio consigliato: {'ANOVA parametrica' if normalita_ok else 'Test di Kruskal-Wallis (non parametrico)'}")

# QQ Plot
plt.figure(figsize=(15, 5))
for i, gruppo in enumerate(gruppi_nomi):
    plt.subplot(1, len(gruppi_nomi), i+1)
    sm.qqplot(gruppi_dati[i], line='s', ax=plt.gca())
    plt.title(f'QQ Plot {var_dipendente} - Gruppo {gruppo}')
plt.tight_layout()
save_figure("02_qqplot_sleep_efficiency_by_group")
print("Figura 2: QQ-Plot di Sleep Efficiency per gruppo salvata")

# Kruskal-Wallis
if len(gruppi_dati) > 1:
    h_stat, p_val = stats.kruskal(*gruppi_dati)
    print(f"\nTest di Kruskal-Wallis per {var_dipendente} tra gruppi:")
    print(f"H = {h_stat:.4f}, p-value = {p_val:.4f}")
    print(f"Differenze significative tra gruppi: {'Sì' if p_val < 0.05 else 'No'}")

    # Post-hoc
    if p_val < 0.05:
        try:
            from scikit_posthocs import posthoc_dunn
            dunn_data = pd.DataFrame({
                'valore': np.concatenate(gruppi_dati),
                'gruppo': np.concatenate([[g] * len(d) for g, d in zip(gruppi_nomi, gruppi_dati)])
            })
            dunn_result = posthoc_dunn(dunn_data, val_col='valore', group_col='gruppo', p_adjust='bonferroni')
            print("\nRisultati test di Dunn (Bonferroni):")
            print(dunn_result)
        except ImportError:
            print("Installa 'scikit-posthocs' con: pip install scikit-posthocs")
            print("\nEsecuzione test Mann-Whitney U:")
            for i in range(len(gruppi_nomi)):
                for j in range(i+1, len(gruppi_nomi)):
                    u_stat, p = stats.mannwhitneyu(gruppi_dati[i], gruppi_dati[j])
                    n_comp = (len(gruppi_nomi)*(len(gruppi_nomi)-1))//2
                    p_adj = min(p * n_comp, 1.0)
                    print(f"{gruppi_nomi[i]} vs {gruppi_nomi[j]}: U = {u_stat:.1f}, p = {p:.4f}, p_adj = {p_adj:.4f}{' (significativo)' if p_adj < 0.05 else ''}")

# Analisi covariate numeriche
print("\nAnalisi delle potenziali covariate:")
covariate_numeriche = ['Age', 'Disease_Duration', 'LEDD', 'UPDRSIII']
correlazioni = []

for idx, cov in enumerate(covariate_numeriche):
    if cov in df.columns:
        dati_validi = df[[var_dipendente, cov]].dropna()
        if len(dati_validi) > 3:
            rho, p = stats.spearmanr(dati_validi[var_dipendente], dati_validi[cov])
            correlazioni.append({'Covariata': cov, 'Correlazione': rho, 'P-value': p})

            # Scatter plot
            plt.figure(figsize=(8, 6))
            sns.scatterplot(data=df, x=cov, y=var_dipendente, hue=var_gruppo)
            plt.title(f'Spearman: {cov} vs {var_dipendente}\nrho = {rho:.2f}, p = {p:.4f}')
            plt.tight_layout()
            save_figure(f"03_{idx+1}_scatter_{cov}_vs_sleep_efficiency")
            print(f"Figura 3.{idx+1}: scatter {cov} vs Sleep Efficiency salvata")

# Tabella correlazioni
if correlazioni:
    correlazioni_df = pd.DataFrame(correlazioni)
    correlazioni_df['Significativa'] = correlazioni_df['P-value'] < 0.05
    print("\nCorrelazioni:")
    print(correlazioni_df)

# Differenze covariate tra gruppi
print("\nDifferenze tra gruppi:")
for idx, cov in enumerate(covariate_numeriche):
    gruppi_cov = [df[df[var_gruppo]==g][cov].dropna() for g in df[var_gruppo].unique()]
    if all(len(g) > 1 for g in gruppi_cov):
        normalita = all(stats.shapiro(g)[1] > 0.05 for g in gruppi_cov if len(g) > 3)
        test = stats.f_oneway if normalita else stats.kruskal
        stat, p_val = test(*gruppi_cov)
        metodo = "ANOVA" if normalita else "Kruskal-Wallis"
        print(f"{cov}: {metodo} p = {p_val:.4f}")

        # Boxplot
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df, x=var_gruppo, y=cov)
        plt.title(f'{cov} per Gruppo\np = {p_val:.4f}')
        plt.tight_layout()
        save_figure(f"04_{idx+1}_boxplot_{cov}_by_group")
        print(f"Figura 4.{idx+1}: Boxplot {cov} per gruppo salvata")

# Covariate categoriche
covariate_categoriche = ['Gender', 'DA_AGONIST_YES1NOT0', 'BDZ_YES1NOT0']
for cov in covariate_categoriche:
    if cov in df.columns:
        contingency = pd.crosstab(df[var_gruppo], df[cov])
        print(f"\nTabella contingenza {cov}:")
        print(contingency)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, dof, expected = stats.chi2_contingency(contingency)
            print(f"Chi2 = {chi2:.4f}, p = {p:.4f}, significativo: {'Sì' if p < 0.05 else 'No'}")

# Identificazione confondenti
print("\nPotenziali fattori confondenti:")
confondenti = [c['Covariata'] for c in correlazioni if c['P-value'] < 0.05]
for cov in covariate_numeriche:
    gruppi_cov = [df[df[var_gruppo]==g][cov].dropna() for g in df[var_gruppo].unique()]
    if all(len(g) > 1 for g in gruppi_cov):
        normalita = all(stats.shapiro(g)[1] > 0.05 for g in gruppi_cov if len(g) > 3)
        test = stats.f_oneway if normalita else stats.kruskal
        _, p_val = test(*gruppi_cov)
        if p_val < 0.05 and cov not in confondenti:
            confondenti.append(cov)

for cov in covariate_categoriche:
    contingency = pd.crosstab(df[var_gruppo], df[cov])
    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        _, p, _, _ = stats.chi2_contingency(contingency)
        if p < 0.05 and cov not in confondenti:
            confondenti.append(cov)

print(f"Fattori confondenti identificati ({len(confondenti)}): {', '.join(confondenti)}")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

# === Caricamento dati ===
df = pd.read_csv("your_dataset.csv")  # Sostituisci con il tuo percorso

# === Creazione cartella salvataggio immagini ===
output_folder = "./confounding"
os.makedirs(output_folder, exist_ok=True)

# === 1. Boxplot Sleep Efficiency per gruppo ===
plt.figure()
sns.boxplot(x='Group', y='Sleep_Efficiency', data=df)
plt.title("Boxplot Sleep Efficiency (%) per Gruppo")
plt.savefig(f"{output_folder}/boxplot_sleep_efficiency.png")
plt.close()
print("Figura 1: Boxplot di Sleep Efficiency per gruppo salvata")

# === 2. Test Shapiro-Wilk per normalità ===
print("\nTest di normalità per gruppo (Shapiro-Wilk):")
normal = True
for group in df['Group'].unique():
    w, p = stats.shapiro(df[df['Group'] == group]['Sleep_Efficiency'])
    norm = "Sì" if p > 0.05 else "No"
    print(f"Gruppo {group}: statistica = {w:.4f}, p-value = {p:.4f}, normalità = {norm}")
    if p <= 0.05:
        normal = False
print(f"\nTutti i gruppi seguono distribuzione normale: {'Sì' if normal else 'No'}")
print(f"Approccio consigliato: {'ANOVA' if normal else 'Test di Kruskal-Wallis (non parametrico)'}")

# === 3. QQ-plot per ogni gruppo ===
for group in df['Group'].unique():
    stats.probplot(df[df['Group'] == group]['Sleep_Efficiency'], dist="norm", plot=plt)
    plt.title(f"QQ-Plot Sleep Efficiency - {group}")
    plt.savefig(f"{output_folder}/qqplot_sleep_efficiency_{group}.png")
    plt.close()
print("Figura 2: QQ-Plot di Sleep Efficiency per gruppo salvata")

# === 4. Kruskal-Wallis Test ===
h, p = stats.kruskal(
    *[df[df['Group'] == g]['Sleep_Efficiency'] for g in df['Group'].unique()]
)
print("\nTest di Kruskal-Wallis per Sleep Efficiency (%) tra gruppi:")
print(f"H = {h:.4f}, p-value = {p:.4f}")
print(f"Differenze significative tra gruppi: {'Sì' if p < 0.05 else 'No'}")

# === 5. Analisi correlazioni con covariate ===
covariates = ['Age', 'Disease_Duration', 'LEDD', 'UPDRSIII']
correlations = []
for cov in covariates:
    r, pval = stats.pearsonr(df[cov], df['Sleep_Efficiency'])
    correlations.append([cov, r, pval, pval < 0.05])
    sns.scatterplot(x=df[cov], y=df['Sleep_Efficiency'])
    plt.title(f"{cov} vs Sleep Efficiency")
    plt.savefig(f"{output_folder}/scatter_{cov}_sleep_efficiency.png")
    plt.close()
print("\nAnalisi delle potenziali covariate:")
for i, cov in enumerate(covariates):
    print(f"Figura 3.{i+1}: Scatter plot di {cov} vs Sleep Efficiency salvata")

corr_df = pd.DataFrame(correlations, columns=["Covariata", "Correlazione", "P-value", "Significativa"])
display(corr_df)

# === 6. ANOVA o Kruskal per covariate numeriche ===
print("\nDifferenze nelle covariate tra gruppi:")
for cov in covariates:
    if stats.shapiro(df[cov])[1] > 0.05:
        f, pval = stats.f_oneway(*[df[df['Group'] == g][cov] for g in df['Group'].unique()])
        test = "ANOVA"
    else:
        f, pval = stats.kruskal(*[df[df['Group'] == g][cov] for g in df['Group'].unique()])
        test = "Kruskal-Wallis"
    sig = "Sì" if pval < 0.05 else "No"
    print(f"{cov}: {test} = {f:.4f}, p-value = {pval:.4f}, differenze significative: {sig}")
    sns.boxplot(x='Group', y=cov, data=df)
    plt.title(f"Boxplot di {cov} per gruppo")
    plt.savefig(f"{output_folder}/boxplot_{cov}.png")
    plt.close()
    print(f"Figura 4.{covariates.index(cov)+1}: Boxplot di {cov} per gruppo salvata")

# === 7. Analisi variabili categoriche ===
categorical = ['Gender', 'DA_AGONIST_YES1NOT0', 'BDZ_YES1NOT0']
confounders = []

for var in categorical:
    contingency = pd.crosstab(df['Group'], df[var])
    chi2, pval, _, expected = stats.chi2_contingency(contingency)
    perc_low_counts = (expected < 5).sum() / expected.size * 100
    sig = "Sì" if pval < 0.05 else "No"
    print(f"\nTabella di contingenza per {var}:")
    display(contingency)
    print(f"Chi-quadrato: {chi2:.4f}, p-value: {pval:.4f}")
    print(f"Associazione significativa: {sig}")
    print(f"Attenzione: {perc_low_counts:.1f}% celle con conteggio atteso < 5")
    if pval < 0.05:
        confounders.append(var)

# === 8. Sintesi fattori confondenti ===
# Covariate numeriche con differenza tra gruppi
if corr_df.loc[1, "P-value"] < 0.05:
    confounders.append("Disease_Duration")
if corr_df.loc[2, "P-value"] < 0.05:
    confounders.append("LEDD")

print("\nPotenziali fattori confondenti identificati:")
for c in confounders:
    print(f"- {c}")
print(f"\nTotale fattori confondenti identificati: {len(confounders)}")
print(f"Fattori confondenti da considerare: {', '.join(confounders)}")

# === 9. Suggerimento per analisi con confondenti ===
print("\nPoiché la variabile dipendente non segue una distribuzione normale in tutti i gruppi,")
print("invece dell'ANCOVA parametrica, si consiglia di considerare una delle seguenti opzioni:")
print("1. Modello lineare generalizzato (GLM) con distribuzione appropriata")
print("2. ANCOVA non parametrica (rank-based)")
print("3. Analisi separata per ciascun fattore confondente")

print(f"\nTutte le figure sono state salvate nella cartella: {os.path.abspath(output_folder)}")


In [ ]:
# Step 3: ANCOVA Non Parametrica (Rank-Based) per Sleep Efficiency
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import os

# Impostazioni grafiche
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# Directory output
output_dir = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/scoring/confounding"
os.makedirs(output_dir, exist_ok=True)

def save_figure(name):
    plt.savefig(os.path.join(output_dir, f"{name}.png"), dpi=300, bbox_inches='tight')
    plt.close()

# Caricamento dati
file_path = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/pazienti_parkinson_filtrati.xlsx"
df = pd.read_excel(file_path)
print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")

# Variabili
var_dip = 'Sleep Efficiency (%)'
var_gruppo = 'Group'
confondenti = ['Disease_Duration', 'LEDD', 'DA_AGONIST_YES1NOT0']

# Check variabili
vars_necessarie = [var_dip, var_gruppo] + confondenti
mancanti = [v for v in vars_necessarie if v not in df.columns]
if mancant:
    print(f"ERRORE: Variabili mancanti: {mancanti}")
else:
    print("Tutte le variabili necessarie sono presenti.")

# Descrizione
print(f"\nStatistiche descrittive per {var_dip} per gruppo:")
display(df.groupby(var_gruppo)[var_dip].describe())

# ANCOVA Non Parametrica
print("\n--- ANCOVA NON PARAMETRICA (RANK-BASED) ---")
df_ranks = df.copy()
df_ranks['Sleep_Efficiency_Rank'] = df[var_dip].rank()

# Modello rank-based
formula = "Sleep_Efficiency_Rank ~ C(Group) + Disease_Duration + LEDD + C(DA_AGONIST_YES1NOT0)"
model = ols(formula, data=df_ranks).fit()
anova_results = sm.stats.anova_lm(model, typ=2)

print("\nRisultati ANCOVA non parametrica:")
display(anova_results)
print("\nCoefficienti del modello:")
display(model.params)

# P-value gruppo
p_group = anova_results.loc["C(Group)", "PR(>F)"]
print(f"\nEffetto del gruppo: p = {p_group:.4f} -> {'Significativo' if p_group < 0.05 else 'Non significativo'}")

# Confondenti
for conf in confondenti:
    conf_term = f"C({conf})" if conf == "DA_AGONIST_YES1NOT0" else conf
    if conf_term in anova_results.index:
        p = anova_results.loc[conf_term, "PR(>F)"]
        print(f"Effetto di {conf}: p = {p:.4f} -> {'Significativo' if p < 0.05 else 'Non significativo'}")

# Post-hoc se significativo
if p_group < 0.05:
    print("\nConfronti post-hoc (solo se gruppo significativo)")
    df_ranks["Adjusted"] = model.fittedvalues
    plt.figure(figsize=(8, 6))
    sns.barplot(x="Group", y="Adjusted", data=df_ranks, ci=95)
    plt.title("Sleep Efficiency (Ranghi aggiustati) per Gruppo")
    save_figure("05_adjusted_sleep_efficiency_by_group")

    # Tukey HSD
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    tukey_result = pairwise_tukeyhsd(df_ranks["Adjusted"], df_ranks["Group"])
    print("\nRisultati Tukey HSD:")
    display(pd.DataFrame(tukey_result._results_table.data[1:], columns=tukey_result._results_table.data[0]))

# Analisi dei residui
residui = model.resid
shapiro_stat, shapiro_p = stats.shapiro(residui)
print(f"\nShapiro-Wilk residui: p = {shapiro_p:.4f} -> {'Normali' if shapiro_p > 0.05 else 'Non normali'}")

plt.figure(figsize=(12, 10))
plt.subplot(2, 2, 1)
sns.histplot(residui, kde=True)
plt.title("Distribuzione residui")

plt.subplot(2, 2, 2)
sm.qqplot(residui, line='s', ax=plt.gca())
plt.title("QQ-Plot residui")

plt.subplot(2, 2, 3)
plt.scatter(model.fittedvalues, residui)
plt.axhline(0, color='red')
plt.title("Residui vs Predetti")

plt.subplot(2, 2, 4)
plt.scatter(range(len(residui)), residui)
plt.axhline(0, color='red')
plt.title("Residui vs Ordine osservazioni")

plt.tight_layout()
save_figure("06_residuals_analysis")

# Conclusioni
print("\n--- CONCLUSIONI ---")
print(f"Variabile analizzata: {var_dip}")
print(f"Fattori confondenti: {', '.join(confondenti)}")
if p_group < 0.05:
    print(f"\nL'effetto del gruppo è significativo (p = {p_group:.4f}) dopo il controllo per i confondenti.")
else:
    print(f"\nL'effetto del gruppo NON è significativo (p = {p_group:.4f}).")
# Passo 1: Trasformazione della variabile dipendente in ranghi
import pandas as pd

# Supponiamo che il dataframe si chiami df e contenga le colonne 'Group' e 'Sleep Efficiency'
df['Sleep_Efficiency_Rank'] = df['Sleep Efficiency (%)'].rank()

# Visualizzare i ranghi
print(df[['Group', 'Sleep Efficiency (%)', 'Sleep_Efficiency_Rank']].head())

# Passo 2: ANCOVA sui ranghi
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# Definiamo il modello ANCOVA, includendo anche i fattori confondenti
model = ols('Sleep_Efficiency_Rank ~ C(Group) + C(DA_AGONIST_YES1NOT0) + Disease_Duration + LEDD', data=df).fit()

# Eseguiamo ANOVA per vedere gli effetti
anova_results = anova_lm(model)
print(anova_results)

# Coefficienti del modello
print("\nCoefficienti del modello:")
print(model.params)

# Effetti del gruppo
group_effect_p_value = anova_results.loc['C(Group)', 'PR(>F)']
print("\nEffetto del gruppo (dopo controllo per i confondenti): p = {:.4f}".format(group_effect_p_value))

# Analisi dei residui
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

residuals = model.resid

# Test di Shapiro-Wilk per verificare la normalità dei residui
shapiro_test_stat, shapiro_p_value = stats.shapiro(residuals)
print(f"\nShapiro-Wilk Test Statistica = {shapiro_test_stat:.4f}, p-value = {shapiro_p_value:.4f}")

# Se i residui sono normalmente distribuiti, plot dei residui
sns.histplot(residuals, kde=True)
plt.title('Distribuzione dei residui')
plt.show()

# Passo 3: Analisi dei ranghi medi per gruppo
rank_means = df.groupby('Group')['Sleep_Efficiency_Rank'].mean()
print("\nRanghi medi per gruppo:")
print(rank_means)

# Passo 4: Modello misto lineare sui ranghi
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# Definiamo il modello misto lineare
mixed_model = mixedlm('Sleep_Efficiency_Rank ~ C(Group) + C(DA_AGONIST_YES1NOT0) + Disease_Duration + LEDD', df, groups=df['Group']).fit()

# Risultati del modello misto
print(mixed_model.summary())

# Effetti del gruppo dal modello misto
print("\nEffetti del gruppo dal modello misto:")
group_effect_mixed_model = mixed_model.fe_params['C(Group)[T.DNV]'], mixed_model.fe_params['C(Group)[T.DYS]']
print(f"C(Group)[T.DNV]: {group_effect_mixed_model[0]:.4f}, p-value = {mixed_model.pvalues['C(Group)[T.DNV]']:.4f}")
print(f"C(Group)[T.DYS]: {group_effect_mixed_model[1]:.4f}, p-value = {mixed_model.pvalues['C(Group)[T.DYS]']:.4f}")
